In [1]:
import hashlib
import logging
import os
import sys
from pathlib import Path

import pandas as pd
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf

# Корень проекта (как в training-скриптах)
GENALM_HOME = os.environ.get(
    "GENALM_HOME",
    "/mnt/20tb/aspeedok",
)
os.environ["GENALM_HOME"] = GENALM_HOME

GENA_LM_ROOT = Path(GENALM_HOME) / "GENA_LM"
EXPR_ROOT = GENA_LM_ROOT / "downstream_tasks" / "expression_prediction"
DESCRIPTIONS_DIR = EXPR_ROOT / "descriptions"

if str(GENA_LM_ROOT) not in sys.path:
    sys.path.insert(0, str(GENA_LM_ROOT))

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("dataset_id")

In [6]:
def _name_and_size(path: str | None, genome_path: str | None = None) -> str:
    """Как ExpressionDataset._name_and_size (для хешей кэша)."""
    if path is None:
        return "None|0"
    p = str(path)
    name = Path(p).name
    try:
        size = os.path.getsize(p)
    except OSError:
        if genome_path and p == genome_path:
            genome_sizes_path = Path(genome_path).expanduser().parent.parent / "genome_sizes.tsv"
            if genome_sizes_path.exists():
                base_dir = genome_sizes_path.parent
                df_sizes = pd.read_csv(genome_sizes_path, sep="\t")
                gp = Path(genome_path).expanduser()
                if not gp.is_absolute():
                    gp = base_dir / gp
                for fp, sz in zip(df_sizes["path"], df_sizes["size"]):
                    if pd.isna(fp) or pd.isna(sz):
                        continue
                    fpath = Path(str(fp)).expanduser()
                    if not fpath.is_absolute():
                        fpath = base_dir / fpath
                    if str(fpath.resolve()) == str(gp.resolve()):
                        return f"{name}|{int(sz)}"
        raise FileNotFoundError(f"File not found for hashing: {p}")
    return f"{name}|{size}"


def _intervals_hash(forward_path: str | None, reverse_path: str | None, genome: str | None) -> str:
    fwd = _name_and_size(forward_path, genome)
    rev = _name_and_size(reverse_path, genome)
    return f"{fwd}|{rev}"


def _hash_prefix(forward_path: str | None, reverse_path: str | None) -> str:
    base = (
        os.path.dirname(forward_path)
        if forward_path
        else os.path.dirname(reverse_path)
    )
    return os.path.join(base, "dataset_hash")


def get_tokens_hash_path(
    *,
    forward_intervals_path: str,
    reverse_intervals_path: str,
    genome: str,
    num_before: int,
    token_len_for_fetch: int = 10,
) -> str:
    m = hashlib.blake2b(digest_size=8)
    for part in ("tokens", _intervals_hash(forward_intervals_path, reverse_intervals_path, genome),
                 _name_and_size(genome), str(num_before)):
        m.update(part.encode("utf-8"))
    if token_len_for_fetch != 10:
        m.update(str(token_len_for_fetch).encode("utf-8"))
    prefix = _hash_prefix(forward_intervals_path, reverse_intervals_path)
    return f"{prefix}.{m.hexdigest()}"


def get_signals_hash_path(
    *,
    forward_intervals_path: str,
    reverse_intervals_path: str,
    targets_path: str,
    genome: str,
    num_before: int,
    gen_max_seq_len: int,
    cell_type_ids: list[str],
    norm_bw: bool = False,
) -> str:
    m = hashlib.blake2b(digest_size=8)
    m.update(b"signals")
    m.update(_intervals_hash(forward_intervals_path, reverse_intervals_path, genome).encode())
    m.update(_name_and_size(targets_path).encode())
    m.update(_name_and_size(genome).encode())
    m.update(str(num_before).encode())
    m.update(str(gen_max_seq_len).encode())
    if norm_bw:
        m.update(b"norm_bw")
    m.update("".join(sorted(cell_type_ids)).encode())
    prefix = _hash_prefix(forward_intervals_path, reverse_intervals_path)
    return f"{prefix}.signal.{m.hexdigest()}"


def get_tpm_hash_path(signals_hash_path: str, hash_prefix: str) -> str:
    suffix = signals_hash_path[len(hash_prefix) + len(".signal.") :]
    return f"{hash_prefix}.tpm.{suffix}"


def get_descriptions_h5_path(
    *,
    targets_path: str,
    text_tokenizer: str,
    text_max_seq_len: int,
) -> Path:
    tokenizer_tag = text_tokenizer.replace("/", "_")
    targets_tag = hashlib.blake2b(
        _name_and_size(targets_path).encode("utf-8"),
        digest_size=8,
    ).hexdigest()
    name = (
        f"{Path(targets_path).name}.{targets_tag}."
        f"{tokenizer_tag}.{text_max_seq_len}.description.h5"
    )
    return DESCRIPTIONS_DIR / name


def merge_shared_params(dataset_cfg: dict, shared_params: dict | None) -> dict:
    """Как merge_default_params_with_dataset_config: shared дополняет dataset."""
    if not shared_params:
        return dict(dataset_cfg)

    def _merge(target: dict, source: dict) -> None:
        for key, value in source.items():
            if key in target:
                if isinstance(value, dict) and isinstance(target[key], dict):
                    _merge(target[key], value)
            else:
                target[key] = value.copy() if isinstance(value, dict) else value

    merged = dict(dataset_cfg)
    _merge(merged, shared_params)
    return merged


def _target_class_name(cfg: dict) -> str:
    t = str(cfg.get("_target_", ""))
    return t.split(".")[-1] if t else ""


def h5_caches_from_config(
    config_path: str,
    prefixes: tuple[str, ...] = ("train_dataset_", "valid_dataset_"),
) -> pd.DataFrame:
    config_path = str(Path(config_path).expanduser().resolve())
    config_dir = str(Path(config_path).parent)
    config_name = Path(config_path).stem

    with initialize_config_dir(config_dir=config_dir, version_base=None):
        cfg = compose(config_name=config_name)

    cfg = OmegaConf.to_container(cfg, resolve=True)
    shared = cfg.get("shared_dataset_params") or {}

    rows: list[dict] = []

    for ds_name, ds_cfg in cfg.items():
        if not any(ds_name.startswith(p) for p in prefixes):
            continue
        if not isinstance(ds_cfg, dict):
            continue

        cls_name = _target_class_name(ds_cfg)
        if cls_name != "ExpressionDataset":
            rows.append({
                "ds_name": ds_name,
                "cache_type": "skipped",
                "h5_id": None,
                "h5_path": None,
                "exists": None,
                "n_cell_types": None,
                "cell_type_ids": None,
                "note": f"_target_={cls_name}",
            })
            continue

        merged = merge_shared_params(ds_cfg, shared)
        forward = merged.get("forward_intervals_path")
        reverse = merged.get("reverse_intervals_path")
        targets_path = merged["targets_path"]
        genome = merged["genome"]
        num_before = int(merged.get("num_before", 512))
        token_len = int(merged.get("token_len_for_fetch", 10))
        gen_max_seq_len = int(merged.get("gen_max_seq_len", 1024))
        text_tokenizer = merged.get("text_tokenizer", "intfloat/multilingual-e5-large-instruct")
        text_max_seq_len = int(merged.get("text_max_seq_len", 1000))
        bw = merged.get("bw") or ""
        tpm = merged.get("tpm") or ""
        norm_bw = bool(merged.get("norm_bw", False))

        df_targets = pd.read_csv(targets_path)
        cell_ids = df_targets["id"].astype(str).tolist()
        prefix = _hash_prefix(forward, reverse)

        def _add_row(cache_type: str, base_path: str) -> None:
            h5_path = f"{base_path}.h5"
            h5_id = Path(base_path).name.split(".")[-1]
            rows.append({
                "ds_name": ds_name,
                "cache_type": cache_type,
                "h5_id": h5_id,
                "h5_path": h5_path,
                "exists": os.path.exists(h5_path),
                "n_cell_types": len(cell_ids),
                "cell_type_ids": ",".join(cell_ids),
                "note": None,
            })

        try:
            tokens_base = get_tokens_hash_path(
                forward_intervals_path=forward,
                reverse_intervals_path=reverse,
                genome=genome,
                num_before=num_before,
                token_len_for_fetch=token_len,
            )
            _add_row("tokens", tokens_base)

            if bw:
                signals_base = get_signals_hash_path(
                    forward_intervals_path=forward,
                    reverse_intervals_path=reverse,
                    targets_path=targets_path,
                    genome=genome,
                    num_before=num_before,
                    gen_max_seq_len=gen_max_seq_len,
                    cell_type_ids=cell_ids,
                    norm_bw=norm_bw,
                )
                _add_row("signals", signals_base)

                if tpm:
                    tpm_base = get_tpm_hash_path(signals_base, prefix)
                    _add_row("tpm", tpm_base)

            desc_path = get_descriptions_h5_path(
                targets_path=targets_path,
                text_tokenizer=text_tokenizer,
                text_max_seq_len=text_max_seq_len,
            )
            rows.append({
                "ds_name": ds_name,
                "cache_type": "descriptions",
                "h5_id": desc_path.stem.split(".")[-2] if "." in desc_path.stem else desc_path.stem,
                "h5_path": str(desc_path),
                "exists": desc_path.exists(),
                "n_cell_types": len(cell_ids),
                "cell_type_ids": ",".join(cell_ids),
                "note": None,
            })
        except FileNotFoundError as e:
            rows.append({
                "ds_name": ds_name,
                "cache_type": "error",
                "h5_id": None,
                "h5_path": None,
                "exists": False,
                "n_cell_types": len(cell_ids),
                "cell_type_ids": ",".join(cell_ids),
                "note": str(e),
            })

    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values(["ds_name", "cache_type"]).reset_index(drop=True)
    return out

In [7]:
CONFIG_PATH = (
    "/mnt/20tb/aspeedok/GENA_LM/downstream_tasks/expression_prediction/configs/final_atacseq_Alex.yaml"
)

df_h5 = h5_caches_from_config(CONFIG_PATH)
df_h5

MissingConfigException: Cannot find primary config 'final_atacseq_Alex'. Check that it's in your config search path.

Config search path:
	provider=hydra, path=pkg://hydra.conf
	provider=main, path=file:///mnt/20tb/aspeedok/GENA_LM/downstream_tasks/expression_prediction/configs
	provider=schema, path=structured://

In [4]:
from IPython.display import display

df_h5 = h5_caches_from_config(CONFIG_PATH)
print(f"rows={len(df_h5)}, cols={len(df_h5.columns)}")
display(df_h5)


rows=0, cols=0


""
